# NOAA Storm Events Raw Data Quality

Storm Events details and the zone-to-county mapping.

The audit queries the current `raw.*` ingestion tables directly; transformed tables are out of scope.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "data" / "housing_predict.duckdb").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
con = duckdb.connect(str(ROOT / "data" / "housing_predict.duckdb"), read_only=True)
TABLES = ["noaa","noaa_mapping"]
KEYS = {"noaa":["event_id"],"noaa_mapping":["state_fips","cz_fips"]}
YEAR_COLUMNS = {"noaa":"year"}
EXPECTED_YEARS = set(range(2010, 2025))
def qi(x): return '"' + x.replace('"', '""') + '"'
def columns(t):
    return con.execute("SELECT column_name FROM information_schema.columns WHERE table_schema='raw' AND table_name=? ORDER BY ordinal_position", [t]).df()["column_name"].tolist()
available = set(con.execute("SELECT table_name FROM information_schema.tables WHERE table_schema='raw'").df()["table_name"])
assert set(TABLES) <= available, f"Missing raw tables: {set(TABLES)-available}"


## Inventory, coverage, keys, and sampled missingness

In [ ]:
inventory, key_checks, coverage, missing = [], [], [], []
for table in TABLES:
    cols = columns(table)
    n = con.execute(f"SELECT count(*) FROM raw.{qi(table)}").fetchone()[0]
    inventory.append({"table": table, "rows": n, "columns": len(cols)})
    key = KEYS[table]
    key_sql = ", ".join(qi(c) for c in key)
    invalid = " OR ".join(f"{qi(c)} IS NULL OR trim(cast({qi(c)} AS VARCHAR))=''" for c in key)
    key_checks.append({
        "table": table, "candidate_key": " + ".join(key),
        "invalid_key_rows": con.execute(f"SELECT count(*) FROM raw.{qi(table)} WHERE {invalid}").fetchone()[0],
        "duplicate_key_groups": con.execute(f"SELECT count(*) FROM (SELECT {key_sql}, count(*) n FROM raw.{qi(table)} GROUP BY {key_sql} HAVING n>1)").fetchone()[0],
    })
    year_col = YEAR_COLUMNS.get(table)
    if year_col:
        years = set(con.execute(f"SELECT DISTINCT try_cast({qi(year_col)} AS INTEGER) FROM raw.{qi(table)} WHERE try_cast({qi(year_col)} AS INTEGER) IS NOT NULL").df().iloc[:,0])
        coverage.append({"table":table, "min_year":min(years), "max_year":max(years), "missing_years":sorted(EXPECTED_YEARS-years), "outside_scope":sorted(years-EXPECTED_YEARS)})
    # Identifier fields plus a bounded sample of wide value fields keeps the audit runnable.
    profile_cols = list(dict.fromkeys([*key, *(["NAME"] if "NAME" in cols else []), *cols[:40]]))
    for col in profile_cols:
        m = con.execute(f"SELECT count(*) FROM raw.{qi(table)} WHERE {qi(col)} IS NULL OR trim(cast({qi(col)} AS VARCHAR))=''").fetchone()[0]
        if m: missing.append({"table":table,"column":col,"missing_rows":m,"missing_pct":round(100*m/n,2)})
display(pd.DataFrame(inventory))
display(pd.DataFrame(coverage))
display(pd.DataFrame(key_checks))
display(pd.DataFrame(missing).sort_values("missing_pct",ascending=False).head(100) if missing else pd.DataFrame())


## Source-specific checks

In [ ]:
display(con.execute("""
SELECT count(*) AS row_count,count(DISTINCT event_id) AS distinct_events,
 count(*) FILTER (WHERE county_fips IS NULL OR trim(county_fips)='') AS unresolved_county_rows,
 count(*) FILTER (WHERE try_cast(total_damage AS DOUBLE)<0) AS negative_damage,
 count(*) FILTER (WHERE try_cast(injuries_direct AS INTEGER)<0) AS negative_injuries,
 count(*) FILTER (WHERE try_cast(deaths_direct AS INTEGER)<0) AS negative_deaths,
 count(*) FILTER (WHERE try_strptime(begin_date_time,'%d-%b-%y %H:%M:%S') IS NULL) AS invalid_begin_dates
FROM raw.noaa WHERE try_cast(year AS INTEGER) BETWEEN 2010 AND 2024
""").df())
display(con.execute("""SELECT mapping_method,mapping_confidence,count(*) AS mappings,sum(try_cast(source_row_count AS BIGINT)) AS source_rows FROM raw.noaa_mapping GROUP BY 1,2 ORDER BY mappings DESC""").df())

## Interpretation and limitations

Event IDs may repeat across geographic segments. Zone records rely on a name-based county mapping, so method and confidence must be retained. Reported damage is incomplete and is not total economic loss.

Displayed failures are investigation items. This notebook does not alter source data.

In [ ]:
con.close()